# GSSC-S2D2 Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BillyChern/GSSC-S2D2/blob/main/examples/quickstart.ipynb)

Five-minute tour of the public API: load the headline checkpoint, sample one frame, visualise the result. Runs on a single GPU (any CUDA 12.x card with >= 12 GB) or CPU (slow).

**What you'll do:**
1. Install GSSC-S2D2 and its `spconv-cu126` companion (~ 2 min).
2. Load the released `gssc_mf/gssc_31k_mf_step40000/model_ema.safetensors` deployment checkpoint.
3. Run a single S2D2 correction step on a SemanticKITTI val frame.
4. Compare base SCPNet vs. S2D2 prediction vs. ground truth on a rare class.

**What you won't need:**
- The full SCPNet predictions tar -- we use a single frame.
- The full synthetic pool -- inference only.
- Multi-GPU -- single GPU is enough.

> Asset URLs (checkpoints, predictions, synthetic pool) are released upon publication; `scripts/download_assets.py` fails loudly with that message until then. See `docs/DATASET.md` for sizes and layout.


## 1. Install

Skip the cell below if running locally and you've already done `uv sync + uv pip install spconv-cu126==2.3.8`.

In [ ]:
import sys, subprocess, os

if 'COLAB_GPU' in os.environ:
    if not os.path.exists('GSSC-S2D2'):
        subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/BillyChern/GSSC-S2D2.git'])
    os.chdir('GSSC-S2D2')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'spconv-cu126==2.3.8'])

sys.path.insert(0, 'src')
import gssc
print(f'GSSC-S2D2 v{gssc.__version__} loaded')

## 2. Resolve assets

Until the public asset URLs go live (see `scripts/download_assets.py`), the cell below will print a `docs/DATASET.md` pointer. If you already have local assets, set `DATA_ROOT` to your data directory.

In [ ]:
from pathlib import Path

DATA_ROOT = Path(os.environ.get('DATA_ROOT', 'data'))
# v1.1.0 nested HF layout: data/checkpoints/<group>/<name>/model_ema.safetensors
# (matches scripts/download_assets.py output and docs/MODEL_ZOO.md)
CKPT      = DATA_ROOT / 'checkpoints' / 'gssc_mf' / 'gssc_31k_mf_step40000' / 'model_ema.safetensors'
FRAME_ID  = '000000'

voxel_path = DATA_ROOT / 'SemanticKITTI' / 'sequences' / '08' / 'voxels' / f'{FRAME_ID}.bin'
scp_path   = DATA_ROOT / 'scpnet_predictions' / '08' / f'{FRAME_ID}_pred.npy'
gt_path    = DATA_ROOT / 'SemanticKITTI' / 'sequences' / '08' / 'voxels' / f'{FRAME_ID}.label'

for p in [CKPT, voxel_path, scp_path, gt_path]:
    if not p.exists():
        raise FileNotFoundError(
            f'Missing asset: {p}. See docs/DATASET.md for download instructions '
            f'(assets released upon publication).'
        )
print('All assets resolved')

## 3. Build the model + S2D2 correction sampler

In [ ]:
import torch, numpy as np
from safetensors.torch import load_file
from gssc.diffusion.multinomial import MultinomialDiffusion3DV2
from gssc.models.s2d2_unet import SceneCompletionUNetSparse
from gssc.inference.generate_predictions import load_lidar_voxels

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# NOTE: "Sparse" in SceneCompletionUNetSparse refers to the auxiliary LiDAR
# encoder (spconv), NOT the denoiser. The released denoiser is a DENSE 3D U-Net
# (Conv3d) with additive/AdaGN conditioning (~35M reproduction).
model = SceneCompletionUNetSparse(
    num_classes=20, base_channels=32, time_emb_dim=128,
    lidar_base_channels=16, lidar_out_channels=32, lidar_in_channels=1,
    no_bev=False, ssc_cond_channels=20, ssc_multiscale=False,
).to(device)

# model_ema.safetensors already holds the EMA deployment weights (paper convention).
state = load_file(str(CKPT))
model.load_state_dict(state, strict=False)
model.train(False)
diffusion = MultinomialDiffusion3DV2(num_classes=20, num_timesteps=100, beta_max=0.1).to(device)
print(f'Loaded EMA deployment weights from {CKPT.name} ({len(state)} tensors)')

## 4. Run a single S2D2 correction step

In [ ]:
import torch.nn.functional as F

lidar = load_lidar_voxels(str(voxel_path)).to(device)
scp   = np.load(scp_path).astype(np.int64)
scp_t = torch.from_numpy(scp).unsqueeze(0).to(device)
scp_oh = F.one_hot(scp_t, 20).float().permute(0, 4, 1, 2, 3)

bev = torch.zeros(1, 256, 256, dtype=torch.long, device=device)
for z in range(scp_t.shape[3] - 1, -1, -1):
    layer = scp_t[0, :, :, z]
    mask = (layer > 0) & (bev[0] == 0)
    bev[0][mask] = layer[mask]

with torch.no_grad():
    pred = diffusion.sample_algo2(
        model, bev, lidar, scpnet_pred=scp_t,
        shape=(1, 256, 256, 32), device=device,
        n_steps=1, show_progress=False, ssc_pred=scp_oh,
    )
pred_train = pred.cpu().numpy()[0]
print(f'Pred range: {pred_train.min()}-{pred_train.max()}')
print(f'Occupancy: {(pred_train > 0).sum() / pred_train.size:.2%}')

## 5. Compare against SCPNet base + ground truth

In [ ]:
from gssc.inference.generate_predictions import LEARNING_MAP_INV

lut = np.zeros(256, dtype=np.int64)
for ti, raw in enumerate(LEARNING_MAP_INV):
    lut[int(raw)] = ti
gt_raw   = np.fromfile(gt_path, dtype=np.uint16).reshape(256, 256, 32)
gt_train = lut[np.clip(gt_raw, 0, 255).astype(np.int64)]

def iou(pred: np.ndarray, gt: np.ndarray, cls: int) -> float:
    inter = ((pred == cls) & (gt == cls)).sum()
    union = ((pred == cls) | (gt == cls)).sum()
    return float(inter / union) if union > 0 else float('nan')

for cls, name in [(8, 'motorcyclist'), (18, 'pole'), (1, 'car')]:
    base = iou(scp, gt_train, cls)
    ours = iou(pred_train, gt_train, cls)
    print(f'  {name:<14s}  SCPNet: {base*100:5.2f}%  S2D2: {ours*100:5.2f}%  Delta {(ours-base)*100:+.2f}')

## What's next

- Full 4 071-frame eval: `python scripts/eval.py eval/val_1step --checkpoint data/checkpoints/gssc_mf/gssc_31k_mf_step40000/model_ema.safetensors` (~ 6 min on H100, expected **38.54 % mIoU**)
- D4 TTA: `python scripts/eval.py eval/val_d4tta --checkpoint data/checkpoints/gssc_mf/gssc_31k_mf_step40000/model_ema.safetensors` (~ 3 h, expected **38.73 % mIoU**)
- BEV second task: `python scripts/eval.py eval/bev_secondary --checkpoint data/checkpoints/bev/bev_perception_net/model.safetensors` (paper Sec. 4, **36.09 % BEV mIoU**)
- Headline retrain: `python scripts/train.py train/31k_mf` (~ 24 h on 1x H100)

See [`docs/REPRODUCIBILITY.md`](../docs/REPRODUCIBILITY.md) for the full reproduction matrix.